# Guía de usuario — `mcp_sinim`

Cliente Python para el **SINIM** (Sistema Nacional de Información Municipal de Chile, [datos.sinim.gov.cl](https://datos.sinim.gov.cl)): ~345 comunas, 2001–2025, 480 variables en 9 áreas.

```bash
pip install mcp-sinim
```

Las secciones 1–2 funcionan **offline** (el catálogo viaja con el paquete); desde la sección 3 se consulta el portal del SINIM (con rate limit cortés de 0,5 s entre requests).

In [ ]:
from mcp_sinim import SINIMClient

client = SINIMClient(corrmon=False)  # corrmon=True → pesos reales (corrección monetaria)

## 1. El catálogo de variables (offline)

480 variables con código, nombre, área, subárea, unidad y fuente.

In [ ]:
catalog = client.catalog()
print(len(catalog), "variables")
catalog.head()

In [ ]:
# Las 9 áreas temáticas:
catalog["area"].sort_values().unique()

## 2. Búsqueda difusa (offline)

Insensible a mayúsculas y acentos: `"educacion"` y `"EDUCACIÓN"` dan lo mismo.

In [ ]:
client.search("patentes municipales", limit=5)[["code", "name", "unit", "score"]]

In [ ]:
client.search("ingresos propios", limit=5)[["code", "name", "area", "score"]]

## 3. Años disponibles (dinámico)

Se descubren del formulario del portal: cuando SINIM publica un año nuevo, aparece aquí sin actualizar el paquete.

In [ ]:
years = client.years()
print(f"{years[0]}–{years[-1]} ({len(years)} años)")

## 4. Descargar datos

`get()` devuelve un panel tidy: una fila por (comuna, año, variable), con `name` y `unit` ya pegados desde el catálogo.

In [ ]:
# 4173 = Ingresos por Patentes Municipales de Beneficio Municipal (M$)
df = client.get("4173", years=years[-3:])
print(df.shape)
df.query("cod_municipio == '13101'")  # SANTIAGO

In [ ]:
# Varias variables a la vez, en formato ancho (una columna por código):
wide = client.get(["4173", "1310"], years=[years[-1]], tidy=False)
wide.head()

## 5. Comunas y regiones

Los ids de región también se descubren dinámicamente (ej.: 131 = Región Metropolitana).

In [ ]:
rm = client.municipios(region="131")
print(len(rm), "comunas en la RM")
client.search_municipios("nunoa", region="131")  # encuentra ÑUÑOA sin acentos

## 6. Caché en disco (opcional)

Con `cache_dir`, la metadata (catálogo y comunas) se guarda en disco y las sesiones siguientes no tocan la red para esas consultas. Los datos de `get()` nunca se cachean.

In [ ]:
# client_cached = SINIMClient(cache_dir="~/.sinim")
client.close()

## 7. Servidor MCP

El mismo paquete expone estas capacidades como tools MCP para cualquier cliente compatible:

```bash
mcp-sinim
```

Tools: `search_variables`, `get_variable_info`, `get_data`, `list_areas`, `list_municipios`, `list_years`. Ver el README para un ejemplo de configuración.